In [9]:
import pandas as pd

In [22]:
gas = pd.read_csv("GNESTE_Gas_Power.csv")
print(gas.head())

  Technology     ID      Continent                   Country ISO2 ISO3  \
0        Gas  G0001  North America  United States of America   US  USA   
1        Gas  G0002  North America  United States of America   US  USA   
2        Gas  G0003  North America  United States of America   US  USA   
3        Gas  G0004  North America  United States of America   US  USA   
4        Gas  G0005  North America  United States of America   US  USA   

       Variable    Code       Unit Category  ... 2019 2020 2021     2022  \
0  Capital Cost   CAPEX     USD/kW     CCGT  ...  NaN  NaN  NaN   650.00   
1  Capital Cost   CAPEX     USD/kW     CCGT  ...  NaN  NaN  NaN  1300.00   
2     Fixed O&M  OPEX_F  USD/kW/yr     CCGT  ...  NaN  NaN  NaN    10.00   
3     Fixed O&M  OPEX_F  USD/kW/yr     CCGT  ...  NaN  NaN  NaN    17.00   
4     Fixed O&M  OPEX_F  USD/kW/yr     CCGT  ...  NaN  NaN  NaN     9.25   

   2023  2024  2025  2030  2040  2050  
0   NaN   NaN   NaN   NaN   NaN   NaN  
1   NaN   NaN   Na

In [23]:
# only keep rows with "European Union" under the country index and relevant data
gas_CCGT = gas[(gas["Country"] == "European Union") & (gas['Variable'].isin(["Fixed O&M", "Lifetime", "Capital Cost"])) & (gas["Category"] == "CCGT")]
print(gas_CCGT)

    Technology     ID Continent         Country ISO2 ISO3      Variable  \
217        Gas  G0218    Europe  European Union  NaN  NaN  Capital Cost   
218        Gas  G0219    Europe  European Union  NaN  NaN  Capital Cost   
221        Gas  G0222    Europe  European Union  NaN  NaN     Fixed O&M   
222        Gas  G0223    Europe  European Union  NaN  NaN     Fixed O&M   
237        Gas  G0238    Europe  European Union  NaN  NaN  Capital Cost   
239        Gas  G0240    Europe  European Union  NaN  NaN     Fixed O&M   
241        Gas  G0242    Europe  European Union  NaN  NaN      Lifetime   

         Code       Unit Category  ... 2019       2020 2021  2022  2023  2024  \
217     CAPEX     USD/kW     CCGT  ...  NaN  872.61840  NaN   NaN   NaN   NaN   
218     CAPEX     USD/kW     CCGT  ...  NaN  993.81540  NaN   NaN   NaN   NaN   
221    OPEX_F  USD/kW/yr     CCGT  ...  NaN   18.17955  NaN   NaN   NaN   NaN   
222    OPEX_F  USD/kW/yr     CCGT  ...  NaN   18.17955  NaN   NaN   NaN   N

In [25]:

gas_CCGT = gas.loc[
    (gas["Country"] == "European Union") & (gas["Category"] == "CCGT")
].copy()

gas_CCGT["Value"] = gas_CCGT.apply(
    lambda row: (row["2030"] * 1e03) / 1.1
    if row["Variable"] in ["Fixed O&M", "Capital Cost"]
    else row["2030"],
    axis=1,
)
print(gas_CCGT)

    Technology     ID Continent         Country ISO2 ISO3      Variable  \
217        Gas  G0218    Europe  European Union  NaN  NaN  Capital Cost   
218        Gas  G0219    Europe  European Union  NaN  NaN  Capital Cost   
221        Gas  G0222    Europe  European Union  NaN  NaN     Fixed O&M   
222        Gas  G0223    Europe  European Union  NaN  NaN     Fixed O&M   
225        Gas  G0226    Europe  European Union  NaN  NaN  Variable O&M   
226        Gas  G0227    Europe  European Union  NaN  NaN  Variable O&M   
229        Gas  G0230    Europe  European Union  NaN  NaN    Efficiency   
230        Gas  G0231    Europe  European Union  NaN  NaN    Efficiency   
237        Gas  G0238    Europe  European Union  NaN  NaN  Capital Cost   
239        Gas  G0240    Europe  European Union  NaN  NaN     Fixed O&M   
241        Gas  G0242    Europe  European Union  NaN  NaN      Lifetime   

           Code       Unit Category  ...        2020 2021 2022  2023  2024  \
217       CAPEX     U

In [27]:
gas_CCGT = gas.loc[
    (gas["Country"] == "European Union") & (gas["Category"] == "CCGT")
].copy()

# get the lifetime
lifetime_map = (
    gas_CCGT.loc[gas_CCGT["Variable"] == "Lifetime", ["Technology", "Country", "Category", "2030"]]
    .rename(columns={"2030": "Lifetime_2030"})
    .drop_duplicates()
)

# attach lifetime to all rows
gas_CCGT = gas_CCGT.merge(
    lifetime_map,
    on=["Technology", "Country", "Category"],
    how="left"
)

# compute Value
gas_CCGT["Value"] = gas_CCGT["2030"]

fixed_om_mask = gas_CCGT["Variable"] == "Fixed O&M"
capital_cost_mask = gas_CCGT["Variable"] == "Capital Cost"

gas_CCGT.loc[fixed_om_mask, "Value"] = (gas_CCGT.loc[fixed_om_mask, "2030"] * 1e03) / 1.1
gas_CCGT.loc[capital_cost_mask, "Value"] = (
    (gas_CCGT.loc[capital_cost_mask, "2030"] * 1e03) / 1.1
) / gas_CCGT.loc[capital_cost_mask, "Lifetime_2030"]
print(gas_CCGT)

   Technology     ID Continent         Country ISO2 ISO3      Variable  \
0         Gas  G0218    Europe  European Union  NaN  NaN  Capital Cost   
1         Gas  G0219    Europe  European Union  NaN  NaN  Capital Cost   
2         Gas  G0222    Europe  European Union  NaN  NaN     Fixed O&M   
3         Gas  G0223    Europe  European Union  NaN  NaN     Fixed O&M   
4         Gas  G0226    Europe  European Union  NaN  NaN  Variable O&M   
5         Gas  G0227    Europe  European Union  NaN  NaN  Variable O&M   
6         Gas  G0230    Europe  European Union  NaN  NaN    Efficiency   
7         Gas  G0231    Europe  European Union  NaN  NaN    Efficiency   
8         Gas  G0238    Europe  European Union  NaN  NaN  Capital Cost   
9         Gas  G0240    Europe  European Union  NaN  NaN     Fixed O&M   
10        Gas  G0242    Europe  European Union  NaN  NaN      Lifetime   

          Code       Unit Category  ... 2021 2022 2023  2024        2025  \
0        CAPEX     USD/kW     CCGT 

In [31]:
fixed_om_mean = gas_CCGT.loc[gas_CCGT["Variable"] == "Fixed O&M", "2030"].mean()
capital_cost_mean = gas_CCGT.loc[gas_CCGT["Variable"] == "Capital Cost", "2030"].mean()
lifetime_mean = gas_CCGT.loc[gas_CCGT["Variable"] == "Lifetime", "2030"].mean()

gas_CCGT_cost_per_hour = (
    (fixed_om_mean * 1e03) / 1.1 / 8760
    + ((capital_cost_mean * 1e03) / 1.1) / lifetime_mean / 8760
)

print("CCGT average cost per hour:", gas_CCGT_cost_per_hour)
print(gas_CCGT_cost_per_hour * 8760)


CCGT average cost per hour: 6.157022554310225
53935.51757575757


In [30]:
# store in csv to be used by the model
fixedom_df = pd.DataFrame({
    "tech": ["CCGT"],
    "fixed_om_eur_MW_hour": [gas_CCGT_cost_per_hour]
})
fixedom_df.to_csv("fixed_om_costs.csv", index=False)